In [ ]:
'''This part is all about the functions that compute style loss, content loss, and total variation loss.
Functions that help visualize the loss transformation graphs are also included here.
'''

In [ ]:
import torch
from torch.nn.functional import mse_loss


def calc_content_loss(features, targets, nodes):
    content_loss = 0
    for node in nodes:
        content_loss += mse_loss(features[node], targets[node])
    return content_loss


def gram(x):
    b, c, h, w = x.size()
    f = x.flatten(2)
    g = torch.bmm(f, f.transpose(1, 2))
    return g.div(h*w)


def calc_style_loss(features, targets, nodes):
    gram_loss = 0
    for node in nodes:
        gram_loss += mse_loss(gram(features[node]), gram(targets[node]))
    return gram_loss

def calc_style_loss_custom(features, targets, nodes):
    gram_loss = 0
    for node in nodes:
        gram_loss += mse_loss(gram(features[node]),  targets[node].expand_as(gram(features[node])))
    return gram_loss


def calc_tv_loss(x): # Total variation loss helps to reduce the noise in the stylized image
    tv_loss = torch.mean(torch.abs(x[:, :, :, :-1] - x[:, :, :, 1:]))
    tv_loss += torch.mean(torch.abs(x[:, :, :-1, :] - x[:, :, 1:, :]))
    return tv_loss

In [ ]:
def save_losses(filepath):
    """Saves the global loss lists to a .pth file."""
    global total_losses, style_losses, content_losses
    loss_data = {
        'total_losses': total_losses,
        'style_losses': style_losses,
        'content_losses': content_losses
    }
    torch.save(loss_data, filepath)
    print(f"Losses saved to {filepath}")

def load_losses(filepath):
    """Loads the global loss lists from a .pth file."""
    global total_losses, style_losses, content_losses
    if os.path.exists(filepath):
        loss_data = torch.load(filepath)
        total_losses = loss_data.get('total_losses', [])
        style_losses = loss_data.get('style_losses', [])
        content_losses = loss_data.get('content_losses', [])
        print(f"Losses loaded from {filepath}")
    else:
        print(f"No loss data found at {filepath}. Starting with empty loss.")


def plot_losses_epoch (total_losses, style_losses, content_losses, loss_type, start_epoch, end_epoch, figsize=(8, 6), fontsize_tick=10, fontsize_label=12, fontsize_title=14):
  sliced_total_losses = total_losses[start_epoch: end_epoch]
  sliced_style_losses = style_losses[start_epoch: end_epoch]
  sliced_content_losses = content_losses[start_epoch: end_epoch]

  epoch_range = range(start_epoch, start_epoch + len(sliced_total_losses))
  plt.figure(figsize=figsize)
  if loss_type == 'all' or loss_type == 'total':
    plt.plot(epoch_range, sliced_total_losses, label = 'Total Loss')
  if loss_type == 'all' or loss_type == 'style':
    plt.plot(epoch_range, sliced_style_losses, label = 'Style Loss')
  if loss_type == 'all' or loss_type == 'content':
    plt.plot(epoch_range, sliced_content_losses, label = 'Content Loss')

  plt.xlabel('Epoch', fontsize=fontsize_label)
  plt.ylabel('Loss', fontsize=fontsize_label)
  plt.legend(fontsize=fontsize_label)
  plt.grid(True)
  plt.title(f'{loss_type} Loss over Epochs (from epoch {start_epoch})', fontsize=fontsize_title)
  plt.tick_params(axis='both', which='major', labelsize=fontsize_tick)
  plt.show()